# Phase 2 — QLoRA Fine-Tuning: Llama-3-8B-Instruct for Civic Complaint Routing

**Runtime required:** Colab Free Tier with a **T4 GPU**.
Go to `Runtime > Change runtime type > T4 GPU` before running anything below.

This notebook:
1. Installs Unsloth + dependencies
2. Loads `unsloth/llama-3-8b-Instruct-bnb-4bit` in 4-bit precision
3. Loads your `train.jsonl` (generated in Phase 1) and formats it for training
4. Attaches LoRA adapters and fine-tunes with **target-only loss masking**
   (loss is computed only on the assistant's JSON output, not the user's input)
5. Runs a quick before/after inference sanity check
6. Pushes the trained LoRA adapter to the Hugging Face Hub for Phase 3

**Before you start:** generate a free Hugging Face **write** access token at
https://huggingface.co/settings/tokens — you'll paste it in Cell 2.


## 1. Install dependencies

This takes ~2-3 minutes on first run.

In [ ]:
%%capture
import os

# Unsloth ships a one-liner that auto-detects the Colab GPU and installs
# the correct compatible versions of torch/xformers/bitsandbytes for you.
!pip install unsloth
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE -- check Runtime > Change runtime type > T4 GPU")


## 2. Hugging Face authentication

Needed to push the trained adapter to the Hub at the end of this notebook.
Get a **write**-scoped token from https://huggingface.co/settings/tokens


In [ ]:
from huggingface_hub import login
from getpass import getpass

hf_token = getpass("Paste your Hugging Face WRITE token: ")
login(token=hf_token)


## 3. Load the base model in 4-bit (QLoRA)

`unsloth/llama-3-8b-Instruct-bnb-4bit` is Unsloth's pre-quantized version of
Meta's Llama-3-8B-Instruct -- free to use, no gated-repo approval needed,
and loads ~4x faster than quantizing the full-precision model yourself.

Loading in 4-bit precision is what makes this fit on a free T4 GPU
(~16GB VRAM) -- the full-precision model would need ~32GB+.


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048   # complaints + JSON labels are short; this is generous headroom
dtype = None             # auto-detect: bfloat16 if supported, else float16
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

print("Model loaded successfully.")


## 4. Attach LoRA adapters

Instead of fine-tuning all 8 billion parameters, we freeze the base model
and only train small low-rank "adapter" matrices injected into specific layers.
This is what makes QLoRA fine-tuning feasible on a single free GPU --
typically well under 1% of the total parameters actually get updated.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                     # LoRA rank -- 16 is a solid default for this size of task
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,            # 0 is optimized/recommended by Unsloth for speed
    bias="none",
    use_gradient_checkpointing="unsloth",  # reduces VRAM usage further
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

model.print_trainable_parameters()


## 5. Load and format the Phase 1 dataset

Upload your `train.jsonl` (from `data/processed/train.jsonl` in Phase 1) to this
Colab session using the file browser on the left, or mount Google Drive --
either works, just adjust the path below.

Each line is already in ChatML format:
```json
{"messages": [{"role": "system", ...}, {"role": "user", ...}, {"role": "assistant", ...}]}
```

We convert this into Llama-3's actual chat template string, which is what
the model needs to see during training.


In [ ]:
from datasets import load_dataset

# Adjust this path if you uploaded the file elsewhere or mounted Drive
DATASET_PATH = "train.jsonl"

dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
print(f"Loaded {len(dataset)} examples")
print(dataset[0])


In [ ]:
def formatting_func(examples):
    """Applies Llama-3's chat template to each ChatML message list."""
    texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        texts.append(text)
    return {"text": texts}


dataset = dataset.map(formatting_func, batched=True)
print(dataset[0]["text"][:800])  # sanity check: should show the formatted chat


## 6. Train with target-only loss masking

This is the key efficiency detail from your project spec: **loss is computed
only on the assistant's JSON output tokens, not on the system prompt or the
user's raw complaint text.** This makes training converge faster and avoids
wasting gradient updates on text the model doesn't need to learn to generate.

Unsloth's `train_on_responses_only` utility handles this masking automatically
by matching the instruction/response template markers.


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,           # adjust based on dataset size; 3 is a safe default for ~1500 rows
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

# Mask the loss so it's computed ONLY on the assistant's JSON response,
# not on the system prompt or the user's raw complaint text.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)


In [ ]:
# Quick VRAM check before the long run
gpu_stats = torch.cuda.get_device_properties(0)
start_mem = torch.cuda.max_memory_reserved() / 1024**3
print(f"GPU = {gpu_stats.name}, max memory = {gpu_stats.total_memory / 1024**3:.1f} GB")
print(f"Memory reserved before training: {start_mem:.2f} GB")


In [ ]:
trainer_stats = trainer.train()


## 7. Sanity-check inference

Before pushing anything to the Hub, run a couple of held-out-style prompts
through the fine-tuned model and eyeball whether the output is clean JSON
matching your schema.


In [ ]:
FastLanguageModel.for_inference(model)  # enables Unsloth's faster inference path

test_complaints = [
    "Bhai master canteen road pe heavy water logging hai, bikes are slipping",
    "There has been no streetlight on MG Road for a week, very unsafe at night",
]

for complaint in test_complaints:
    messages = [
        {
            "role": "system",
            "content": (
                "You are a civic complaint structuring engine. Convert the user's "
                "message into a single valid JSON object matching the schema. "
                "Output ONLY the JSON object -- no explanations, no markdown."
            ),
        },
        {"role": "user", "content": complaint},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(input_ids=inputs, max_new_tokens=200, use_cache=True)
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

    print(f"INPUT:  {complaint}")
    print(f"OUTPUT: {response}")
    print("-" * 80)


## 8. Push the LoRA adapter to Hugging Face Hub

This uploads ONLY the small LoRA adapter weights (a few hundred MB at most),
not the full 8B base model -- Phase 3 will load the base model fresh from
Unsloth/Hugging Face and attach this adapter on top of it.

**Replace `your-username/civic-complaint-router-lora` with your actual HF username.**


In [ ]:
HF_REPO_ID = "your-username/civic-complaint-router-lora"  # <-- CHANGE THIS

model.push_to_hub(HF_REPO_ID, token=hf_token)
tokenizer.push_to_hub(HF_REPO_ID, token=hf_token)

print(f"Adapter pushed to: https://huggingface.co/{HF_REPO_ID}")


## Done — Phase 2 complete

Your LoRA adapter is now on the Hugging Face Hub. In Phase 3, `src/model_loader.py`
will load the base `unsloth/llama-3-8b-Instruct-bnb-4bit` model and attach this
adapter for production inference inside the LangChain + Pydantic validation pipeline.

**Note the repo ID you used above** -- you'll need it for Phase 3.
